# Demo dóng hàng **Phó Thanh Chủ Nữ Khoa** — Bertalign + Vecalign + API CLC

**Tài liệu chuẩn:** `Hướng Dẫn Dóng Hàng câu Dịch nghĩa Hán - Việt.pdf`. Dóng hàng *dịch nghĩa*, giữ nguyên văn bản nguồn, TSV/XLSX 2 cột và `notes.xlsx` để ghi sai khác (không tự sửa bản dịch).

Notebook chạy **Save Version / Run All** từ đầu. Không cần GPU; Internet cần để cài Bertalign/Vecalign và tải model lần đầu. API CLC có thể chạy trực tiếp hoặc **REPLAY** từ `api_response.json` trước đó mà không gọi API thêm.

**Cần Add Input:** hai TXT `han_test.txt`, `vi_test.txt`, hai module đã cập nhật `HanVietAligner.py` và `EnsembleHanVietAligner.py` (trong cùng `alignment_tool/`), và tùy chọn `api_response.json`.


## 1. CẤU HÌNH — sửa đường dẫn tại đây

`MODULE_DIR` phải trỏ vào thư mục chứa **cả hai module phiên bản đã cập nhật**, không dùng ZIP. Nếu `API_RESPONSE_PATH=None`, notebook gọi API CLC thật; nếu chỉ định JSON thì chạy lại chính response đó và **kiểm tra khớp nguyên văn** với 2 TXT trước khi voting.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

# === HAI FILE TEST (SỬA NẾU CẦN) ===
HAN_PATH = Path('/kaggle/input/dong-hang-data/han_test.txt')
VI_PATH  = Path('/kaggle/input/dong-hang-data/vi_test.txt')

# === THƯ MỤC CHỨA CẢ HAI FILE .py ĐÃ CẬP NHẬT (KHÔNG DÙNG ZIP) ===
MODULE_DIR = Path('/kaggle/input/data-nlp/NLP/NLP-module/alignment_tool')

# === TÙY CHỌN DÙNG LẠI PHẢN HỒI API ĐÃ LƯU (KHÔNG GỌI API) ===
# Ví dụ: Path('/kaggle/input/api-debug/api_response.json')
API_RESPONSE_PATH = None

# === THƯ MỤC TẠM (KHÔNG XUẤT VÀO OUTPUT) ===
TEMP_DIR = Path('/kaggle/temp/nu_khoa_voting')

# === OUTPUT CHỈ CHỨA KẾT QUẢ ===
OUTPUT_DIR = Path('/kaggle/working/nu_khoa_vote_demo')

# Bật API thật hoặc API replay để đủ ba phương pháp.
ENABLE_API = True

print('Đã cấu hình. Kiểm tra đường dẫn ở cell tiếp theo.')


In [ ]:
def find_input(configured: Path, expected_name: str, *, optional: bool = False):
    if configured.is_file():
        return configured
    matches = sorted(p for p in INPUT_ROOT.rglob(expected_name) if p.is_file())
    if len(matches) == 1:
        print(f'[AUTO] {expected_name}: {matches[0]}')
        return matches[0]
    if optional and not matches:
        return None
    print(f'Không xác định được duy nhất {expected_name}; các file tìm thấy:')
    for candidate in matches:
        print(' -', candidate)
    raise FileNotFoundError(f'Hãy sửa đường dẫn {expected_name} ở cell CẤU HÌNH.')

HAN_PATH = find_input(HAN_PATH, 'han_test.txt')
VI_PATH = find_input(VI_PATH, 'vi_test.txt')
assert HAN_PATH.resolve() != VI_PATH.resolve(), 'Không được chọn cùng một file cho hai ngôn ngữ.'

if not all((MODULE_DIR / x).is_file() for x in ('HanVietAligner.py', 'EnsembleHanVietAligner.py')):
    matches = [p.parent for p in INPUT_ROOT.rglob('EnsembleHanVietAligner.py')
               if (p.parent / 'HanVietAligner.py').is_file()]
    assert len(matches) == 1, f'Chọn lại MODULE_DIR. Các thư mục hợp lệ: {matches}'
    MODULE_DIR = matches[0]
    print('[AUTO] MODULE_DIR:', MODULE_DIR)

if API_RESPONSE_PATH is not None:
    API_RESPONSE_PATH = find_input(Path(API_RESPONSE_PATH), 'api_response.json')

print('HAN:', HAN_PATH)
print('VI:', VI_PATH)
print('MODULE:', MODULE_DIR)
print('API mode:', 'REPLAY' if API_RESPONSE_PATH else 'LIVE' if ENABLE_API else 'DISABLED')


## 2. Cài thư viện

Cần Internet để cài Bertalign/Vecalign và tải mô hình embedding lần đầu. **Không được thay engine bị lỗi bằng vote giả.** Nếu thiếu một engine, notebook chỉ xuất kết quả `PARTIAL` và thông báo lỗi.

In [ ]:
import sys, subprocess, importlib.util
print('Python:', sys.version)

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('openpyxl==3.1.5', 'requests>=2.31,<3', 'Cython>=3.0.11', 'sentence-transformers>=3.2,<6')

# Bertalign gốc (phù hợp với lớp adapter hiện tại trong EnsembleHanVietAligner.py).
# Nếu Kaggle đã có bản package phù hợp, tránh tải lại.
if importlib.util.find_spec('bertalign') is None:
    pip('faiss-cpu>=1.8', 'sentence-splitter==1.4', 'numba>=0.60', 'googletrans==4.0.2')
    pip('--no-deps', 'git+https://github.com/bfsujason/bertalign.git')

# Vecalign chính chủ, cần build extension native.
if importlib.util.find_spec('vecalign') is None:
    pip('--no-build-isolation', 'git+https://github.com/thompsonb/vecalign.git')

print('Đã cài xong các thư viện cần thiết.')

## 3. Import trực tiếp module từ Kaggle Input

Không giải nén ZIP, không copy source code vào `/kaggle/working`. **Bắt buộc dùng phiên bản mới** có quy đổi API theo văn bản, vì API có thể tự tách lại 7/13 đơn vị thành 9/21 đơn vị.


In [ ]:
import sys
import inspect

sys.path.insert(0, str(MODULE_DIR))

from HanVietAligner import API_RESEGMENTATION_MAPPING_VERSION
from EnsembleHanVietAligner import EnsembleHanVietAligner, split_sentences

assert API_RESEGMENTATION_MAPPING_VERSION >= 1, 'HanVietAligner.py chưa được cập nhật.'
assert 'api_response' in inspect.signature(EnsembleHanVietAligner.align_section).parameters, (
    'EnsembleHanVietAligner.py chưa được cập nhật phần API replay.'
)
print('Import OK:', MODULE_DIR)
print('CLC mapping version:', API_RESEGMENTATION_MAPPING_VERSION)


## 4. Đọc dữ liệu thật và kiểm tra phạm vi

Hai TXT đã là một tiểu mục, vì vậy **không tìm/chặt theo mốc chương tiếp theo**. Các tiêu đề được giữ làm metadata, không đưa vào danh sách câu để tránh ghép tiêu đề với câu văn.

In [ ]:
import re, unicodedata, hashlib, json

def read_source(path):
    return path.read_text(encoding='utf-8-sig').replace('\r\n', '\n').replace('\r', '\n')

def nonempty_lines(s):
    return [line.strip() for line in s.splitlines() if line.strip()]

han_source = read_source(HAN_PATH)
vi_source = read_source(VI_PATH)
h_lines = nonempty_lines(han_source)
v_lines = nonempty_lines(vi_source)

assert len(h_lines) >= 5 and len(v_lines) >= 5, 'Một trong hai TXT quá ngắn'
assert h_lines[0] == '調經' and h_lines[1] == '經水先期', f'Mốc Hán không đúng: {h_lines[:2]}'
assert 'ĐIỀU KINH' in v_lines[0].upper(), f'Tiêu đề Việt không đúng: {v_lines[:2]}'
assert 'TIÊN KỲ' in v_lines[1].upper(), f'Tiêu đề mục Việt không đúng: {v_lines[:2]}'

print('Hán:', len(h_lines), 'dòng có nội dung;', len(han_source), 'ký tự')
print('Việt:', len(v_lines), 'dòng có nội dung;', len(vi_source), 'ký tự')
print('Tiêu đề Hán:', h_lines[:2])
print('Tiêu đề Việt:', v_lines[:2])
print("\nLưu ý: bản Việt ghi 'KINH THÙY TIÊN KỲ' trong file test; không tự sửa OCR/bản dịch.")

## 5. Tách đơn vị ngữ nghĩa để dóng hàng

- Câu văn xuôi: dùng dấu câu **làm gợi ý**, không coi là quy tắc tuyệt đối (theo tài liệu chuẩn).
- Danh sách bài thuốc: gom **từ tên phương thuốc đến hết cách sắc/uống** thành một khối mỗi bản, do thứ tự/liều lượng thể hiện khác nhau. Không dịch, không sắp xếp lại danh sách thuốc.
- Tiêu đề `調經 / 經水先期` và `1. ĐIỀU KINH / KINH THÙY TIÊN KỲ` chỉ lưu metadata, không tính là câu dóng hàng.

In [ ]:
def find_line(lines, needle):
    hits = [i for i, s in enumerate(lines) if needle in s]
    if len(hits) != 1:
        raise ValueError(f'Cần tìm đúng một dòng {needle!r}; được {hits}. Kiểm tra TXT.')
    return hits[0]

def prose_units(lines, lang):
    out = []
    for line in lines:
        units = split_sentences(line, lang)
        out.extend(units)
    return out

# Vị trí danh sách thuốc trong hai file.
h_med_start = find_line(h_lines, '方用清經散')
h_med_end   = find_line(h_lines, '二劑而火自平')
v_med_start = find_line(v_lines, 'Phương thuốc này gọi là:')
v_med_end   = find_line(v_lines, 'Lấy nước sắc uống, uống 2 thang')

assert 2 < h_med_start <= h_med_end < len(h_lines) - 1
assert 2 < v_med_start <= v_med_end < len(v_lines) - 1

han_intro = prose_units(h_lines[2:h_med_start], 'han')
vi_intro = prose_units(v_lines[2:v_med_start], 'vi')
han_recipe = ' '.join(h_lines[h_med_start:h_med_end + 1])
vi_recipe = ' '.join(v_lines[v_med_start:v_med_end + 1])
han_tail = prose_units(h_lines[h_med_end + 1:], 'han')
vi_tail = prose_units(v_lines[v_med_end + 1:], 'vi')

han_sents = han_intro + [han_recipe] + han_tail
vi_sents = vi_intro + [vi_recipe] + vi_tail

# Không bỏ sót dòng có nội dung chính, không thay đổi từ/câu nguồn.
for body_line in h_lines[2:]:
    assert body_line in ' '.join(han_sents), f'Mất nội dung Hán: {body_line!r}'
for body_line in v_lines[2:]:
    assert body_line in ' '.join(vi_sents), f'Mất nội dung Việt: {body_line!r}'

assert len(han_sents) > 2 and len(vi_sents) > 2
assert all('\n' not in s and '\t' not in s for s in han_sents + vi_sents)
assert len(han_sents) <= 80 and len(vi_sents) <= 80, 'Quá giới hạn đơn vị API cho demo'

han_recipe_id = len(han_intro)
vi_recipe_id = len(vi_intro)
print('Hán:', len(han_sents), 'đơn vị | khối thuốc H', han_recipe_id)
print('Việt:', len(vi_sents), 'đơn vị | khối thuốc V', vi_recipe_id)
print('Cảnh báo: khối thuốc cần đối chiếu thủ công về thứ tự/tên/liều lượng.')

In [ ]:
from IPython.display import display
import pandas as pd

print('=== HÁN: các đơn vị đầu vào ===')
display(pd.DataFrame([{'id': i, 'loại': 'BÀI THUỐC' if i == han_recipe_id else 'VĂN XUÔI', 'nội dung': s}
                      for i, s in enumerate(han_sents)]))
print('=== VIỆT: các đơn vị đầu vào ===')
display(pd.DataFrame([{'id': i, 'loại': 'BÀI THUỐC' if i == vi_recipe_id else 'VĂN XUÔI', 'nội dung': s}
                      for i, s in enumerate(vi_sents)]))

# DỪNG VÀ KIỂM TRA: nếu nhận thấy còn lỗi OCR, sai phạm vi, thiếu nội dung,
# chỉnh lại file đầu vào (giữ nguyên bản gốc) trước khi chạy ba engine.

## 6. API CLC (Live hoặc Replay)

- **LIVE:** nếu không có `API_RESPONSE_PATH` thì gọi API, đọc `CLC_API_TOKEN` từ Kaggle Secrets (nếu có).
- **REPLAY:** nếu có JSON đã lưu thì không gọi mạng/token; module xác minh văn bản API khớp input đã phân đoạn rồi mới tính vote. Phù hợp để kiểm tra lại kết quả và không dùng quota.


In [ ]:
import os
API_TOKEN = os.environ.get('CLC_API_TOKEN', '')

if ENABLE_API and API_RESPONSE_PATH is None and not API_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        API_TOKEN = UserSecretsClient().get_secret('CLC_API_TOKEN') or ''
    except Exception:
        API_TOKEN = ''

print('API:', 'REPLAY (không gọi mạng/token)' if API_RESPONSE_PATH else
      'LIVE' if ENABLE_API else 'DISABLED')
if ENABLE_API and API_RESPONSE_PATH is None and not API_TOKEN:
    print('Chưa có token. Nếu API yêu cầu xác thực, engine API sẽ lỗi và output là PARTIAL.')


## 7. Chạy Bertalign + Vecalign + API CLC và voting

**Tất cả phương pháp dùng cùng danh sách câu.** Native index API được ánh xạ lại theo `han_text`/`vietnamese_text` và kiểm tra toàn văn, sau đó mới voting. Nếu chỉ số API bị lệch hoặc response khác văn bản test, module dừng API, không tự đoán.


In [ ]:
aligner = EnsembleHanVietAligner(
    api_token=API_TOKEN or None,
    vec_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    overlap_n=4,
    batch_size=16,
)
result = aligner.align_section(
    han_sents,
    vi_sents,
    out_dir=TEMP_DIR / 'results',
    enable_bertalign=True,
    enable_vecalign=True,
    enable_api=ENABLE_API,
    api_response=API_RESPONSE_PATH,  # None = gọi API thật
)
print('Engine thành công:', result.methods_ok)
print('Lỗi:', result.method_errors)
print('Trạng thái:', result.stats.get('status'))
print('Số cặp voting được chọn:', result.stats.get('accepted_pairs'))
print('Số dòng cần review:', result.stats.get('review_items'))


## 8. Xuất **chỉ kết quả** vào `/kaggle/working`

Không sao chép TXT gốc, thư mục code, JSON replay, embeddings hay overlaps. Giữ `input_sentences.json` (đầu vào đã phân đoạn), `api_mapping.json` (bằng chứng ánh xạ API), đề xuất của ba phương pháp, voting, review, báo cáo, TSV/XLSX và `notes.xlsx`.


In [ ]:
import shutil
from openpyxl import load_workbook

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keep = {
    'input_sentences.json',
    'bertalign_proposals.json',
    'vecalign_proposals.json',
    'api_proposals.json',
    'api_mapping.json',
    'vote_details.json',
    'review.jsonl',
    'report.json',
    'notes.xlsx',
    'nu_khoa_parallel.tsv',
    'nu_khoa_parallel.xlsx',
    'nu_khoa_parallel.PARTIAL.tsv',
    'nu_khoa_parallel.PARTIAL.xlsx',
}
# Xoá các file kết quả của lần chạy cũ để tránh trộn trạng thái.
for name in keep | {'demo_metadata.json'}:
    previous = OUTPUT_DIR / name
    if previous.is_file():
        previous.unlink()

for p in result.output_dir.iterdir():
    if p.is_file() and p.name in keep:
        shutil.copy2(p, OUTPUT_DIR / p.name)

# Các lỗi/chỗ cần rà soát trong bản dịch đã xuất bản: chỉ ghi chú, không sửa nội dung.
notes_file = OUTPUT_DIR / 'notes.xlsx'
wb = load_workbook(notes_file)
ws = wb['NOTES']
ws.append([
    'Phó Thanh Chủ Nữ Khoa', '方用清經散。', 'THANH KINH THANG',
    'Nghi vấn khác tên/dạng phương thuốc: 清經散 (tán) so với THANG (thang).',
    'Đối chiếu bản in/nguồn; không sửa bản dịch gốc.',
])
ws.append([
    'Phó Thanh Chủ Nữ Khoa', '地骨皮（五錢）', 'ĐỊA CỐT BÌ (bỏ xương) 2 đồng cản.',
    'Nghi vấn khác liều lượng (Hán: 五錢; Việt: 2 đồng cân theo bản hiện có).',
    'Kiểm tra lại cả OCR và bản dịch; ghi nhận sai khác, không tự sửa.',
])
ws.append([
    'Phó Thanh Chủ Nữ Khoa', han_recipe, vi_recipe,
    'Thứ tự các vị thuốc không giống nhau; đây là lý do gộp khối thuốc khi test.',
    'Rà soát thủ công từng vị trước khi quyết định dữ liệu nộp.',
])
wb.save(notes_file)

metadata = {
    'work': 'Phó Thanh Chủ Nữ Khoa — Kinh thủy tiên kỳ',
    'guideline': 'Hướng Dẫn Dóng Hàng câu Dịch nghĩa Hán - Việt.pdf',
    'han_filename': HAN_PATH.name,
    'vi_filename': VI_PATH.name,
    'han_sha256': hashlib.sha256(HAN_PATH.read_bytes()).hexdigest(),
    'vi_sha256': hashlib.sha256(VI_PATH.read_bytes()).hexdigest(),
    'han_headers': h_lines[:2],
    'vi_headers': v_lines[:2],
    'han_recipe_unit_id': han_recipe_id,
    'vi_recipe_unit_id': vi_recipe_id,
    'segmentation': 'prose sentences + one recipe block per language',
    'warning': 'Recipe order/dosages require manual checking; voting != semantic validation.',
    'engines_ok': result.methods_ok,
    'api_mode': 'REPLAY' if API_RESPONSE_PATH else 'LIVE' if ENABLE_API else 'DISABLED',
}
(OUTPUT_DIR / 'demo_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Các FILE KẾT QUẢ đã xuất (không có bản sao input):')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(' -', p.name)


## 9. Xem voting và chất lượng đầu ra

Đọc ít nhất 10–20 cặp để kiểm tra ngữ nghĩa; đặc biệt phải đối chiếu tên và liều lượng các vị thuốc, chữ OCR sai/thiếu. Những cặp chưa có đồng thuận hoặc bị bỏ sót cần đưa vào kiểm tra thủ công. Nếu thiếu engine, TSV/XLSX được đánh dấu `PARTIAL`, không được coi là đầu ra 3-way voting hoàn chỉnh.

In [ ]:
from IPython.display import display
print('=== TRẠNG THÁI ===')
print(json.dumps(result.stats, ensure_ascii=False, indent=2))

print('\n=== VOTING ĐƯỢC CHỌN (chưa được xác minh đúng nghĩa) ===')
df_accepted = pd.DataFrame(result.accepted)
if df_accepted.empty:
    print('Không có cặp nào được chọn hoặc chưa đủ 3 engine thành công.')
else:
    display(df_accepted[['han', 'viet', 'votes', 'voters']].head(25))

print('\n=== REVIEW ===')
review_df = pd.DataFrame(result.review)
if review_df.empty:
    print('Không có mục review do module ghi nhận (vẫn cần kiểm tra thủ công).')
else:
    display(review_df.head(30))

print('\nFile notes.xlsx:', OUTPUT_DIR / 'notes.xlsx')

## 10. Tiêu chí đánh giá sau thử nghiệm

1. **Tính hợp lệ:** Bertalign, Vecalign, API đều chạy thành công? Nếu không, nêu rõ lỗi và không coi là kết quả voting đủ ba phương pháp.
2. **Đối chiếu nội dung:** Kiểm tra nhóm 1–1, 1–n, n–1, n–m được ghép đủ nghĩa; không cắt cơ học ở vị trí làm mất quan hệ dịch.
3. **Sai khác bản dịch:** ghi `notes.xlsx` với 5 cột đúng hướng dẫn (tác phẩm, Hán, Việt, lỗi, chỉnh sửa/đề xuất); **không chỉnh sửa văn bản Việt hiện có**.
4. **Kết quả nộp:** TSV hai cột phân cách TAB, mỗi cặp một dòng; Excel sheet `DONG_HANG` đúng hai cột `Hán`, `Việt`. Đầu ra phải được kiểm tra và đạt yêu cầu chất lượng trước khi nộp.
5. **Báo cáo:** dùng `report.json`, `review.jsonl`, ba file proposals và `demo_metadata.json` để giải thích cách chọn cặp, thông tin nguồn, tỷ lệ bao phủ và sai khác cần sửa liên kết.

**Lưu ý:** khi xử lý toàn bộ sách, cần thống nhất phạm vi chương ở hai bản; demo ở đây đã là một tiểu mục tương ứng. Số cặp được ≥2/3 mô hình đồng ý không phải tỷ lệ chính xác 95%.